# Eksperimen Customer Segmentation — K-Means Clustering
**Oleh:** Ahul (IslahulHadi)

Proyek ini melakukan Customer Segmentation menggunakan algoritma K-Means Clustering pada dataset Online Retail.
Segmentasi dilakukan berdasarkan analisis RFM (Recency, Frequency, Monetary).

In [ ]:
# Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 1. LOAD DATA

In [ ]:
# Load data
df = pd.read_csv('OnlineRetail.csv', encoding='latin1')

print("5 baris Pertama")
display(df.head())

print("\nINFO DATA:")
print(df.info())

print("\nDESKRIPSI STATISTIK:")
display(df.describe())

In [ ]:
# Missing Values & Duplikasi
print("Missing Value:")
print(df.isnull().sum())

print("\nJumlah Duplikasi:", df.duplicated().sum())

In [ ]:
# Cek Nilai Unik di Kolom Penting
print("JUMLAH Nilai Unik: ")
print("Jumlah Invoice unik (transaksi berbeda):", df['InvoiceNo'].nunique())
print("Jumlah Customer unik (jumlah customer):", df['CustomerID'].nunique())
print("Jumlah Produk unik (variasi produk):", df['Description'].nunique())
print("Jumlah Kode Produk unik (variasi StockCode):", df['StockCode'].nunique())
print("Jumlah Negara unik:", df['Country'].nunique())

In [ ]:
# Datetime Analysis
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Range Waktu Transaksi:")
print("Transaksi Pertama:", df['InvoiceDate'].min())
print("Transaksi Terakhir:", df['InvoiceDate'].max())

In [ ]:
# Ekstraksi waktu
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour
df['Date'] = df['InvoiceDate'].dt.date

# Analisis transaksi paling ramai
trans_per_hari = df.groupby('Date')['InvoiceNo'].count()
trans_per_bulan = df.groupby('Month')['InvoiceNo'].count()
trans_per_jam = df.groupby('Hour')['InvoiceNo'].count()

print("Tanggal paling ramai:", trans_per_hari.idxmax(), "dengan", trans_per_hari.max(), "transaksi")
print("Bulan paling ramai:", trans_per_bulan.idxmax(), "dengan", trans_per_bulan.max(), "transaksi")
print("Jam paling ramai:", trans_per_jam.idxmax(), "dengan", trans_per_jam.max(), "transaksi")

In [ ]:
# Perhitungan Pendapatan (Revenue)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['Revenue'] = df['Revenue'].round(2)

total_revenue = df['Revenue'].sum()
print("Total Revenue:", f"{total_revenue:,.2f}")

In [ ]:
# Negara Terbanyak Transaksi
top_country = df['Country'].value_counts().head(10)

plt.figure(figsize=(8,4))
sns.barplot(x=top_country.values, y=top_country.index, palette="viridis")
plt.title("Top 10 Negara dengan Transaksi Terbanyak")
plt.xlabel("Jumlah Transaksi")
plt.ylabel("Negara")
plt.show()

In [ ]:
# Top Product berdasarkan revenue
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_products.values, y=top_products.index, palette="magma")
plt.title("Top 10 Produk Berdasarkan Revenue")
plt.xlabel("Revenue")
plt.ylabel("Produk")
plt.show()

In [ ]:
# Analisis Customer
# Customer paling aktif
active_customer = df.groupby('CustomerID')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)
print("Top 10 Customer Paling Aktif:")
display(active_customer)

# Customer dengan pembelian terbanyak
top_cust_qty = df.groupby('CustomerID')['Quantity'].sum().sort_values(ascending=False).head(10)
print("\nTop 10 Customer dengan Quantity Terbanyak:")
display(top_cust_qty)

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df['Quantity'], color='skyblue')
plt.title("Distribusi Quantity")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=df['UnitPrice'], color='salmon')
plt.title("Distribusi UnitPrice")
plt.show()

## 2. PREPROCESSING

### Drop Missing Value

In [ ]:
df_drop = df.dropna(subset=['CustomerID'])
df_drop['CustomerID'] = df_drop['CustomerID'].astype(int)

### Drop Duplicate

In [ ]:
df_duplicate = df_drop.drop_duplicates()

print('Missing value: ', df_duplicate.isnull().sum())
print('Duplikasi: ', df_duplicate.duplicated().sum())

### Bersihkan Data Invalid

In [ ]:
df_step1 = df_duplicate[~df_duplicate['InvoiceNo'].astype(str).str.startswith('C')]

# Remove quantity <= 0
df_clean = df_step1[df_step1['Quantity'] > 0]

# Remove UnitPrice <= 0
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Remove extreme outlier
Q1_Q = df_clean['Quantity'].quantile(0.25)
Q3_Q = df_clean['Quantity'].quantile(0.75)
IQR_Q = Q3_Q - Q1_Q

lower_Q = Q1_Q - 1.5 * IQR_Q
upper_Q = Q3_Q + 1.5 * IQR_Q

Q1_P = df_clean['UnitPrice'].quantile(0.25)
Q3_P = df_clean['UnitPrice'].quantile(0.75)
IQR_P = Q3_P - Q1_P

lower_P = Q1_P - 1.0 * IQR_P
upper_P = Q3_P + 1.0 * IQR_P

df_final = df_clean[
    (df_clean['Quantity'] >= lower_Q) & (df_clean['Quantity'] <= upper_Q) &
    (df_clean['UnitPrice'] >= lower_P) & (df_clean['UnitPrice'] <= upper_P)
]

df_final[['Quantity', 'UnitPrice']].describe()

In [ ]:
plt.figure(figsize=(10,5))

plt.subplot(121)
sns.boxplot(df_final['Quantity'])
plt.title("Distribusi Quantity Setelah Cleaning")

plt.subplot(122)
sns.boxplot(df_final['UnitPrice'])
plt.title("Distribusi UnitPrice Setelah Cleaning")

plt.tight_layout()
plt.show()

### Konversi InvoiceDate menjadi tipe datetime

Agar bisa menghitung Recency di RFM dan mengurutkan data berdasarkan waktu

In [ ]:
df_final1 = df_final.copy()
df_final1['InvoiceDate'] = pd.to_datetime(df_final1['InvoiceDate'])

df_final1['InvoiceDate'].dtype

### Membuat Fitur TotalAmount

In [ ]:
df_final2 = df_final1.copy()
df_final2['TotalAmount'] = df_final2['Quantity'] * df_final2['UnitPrice']

df_final2[['Quantity', 'UnitPrice', 'TotalAmount']].head()

df_final2['TotalAmount'].describe()

print("Jumlah customer sebelum cleaning:", df['CustomerID'].nunique())
print("Jumlah customer setelah cleaning:", df_final2['CustomerID'].nunique())

### Membangun RFM (Recency, Frequency, Monetary)

In [ ]:
# Tanggal referensi
ref_date = df_final2['InvoiceDate'].max()
ref_date

In [ ]:
# Tabel RFM
rfm = df_final2.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (ref_date - x.max()).days,    # Recency
    'InvoiceNo': 'nunique',                                # Frequency
    'TotalAmount': 'sum'                                   # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

In [ ]:
rfm.describe()

### Normalisasi RFM

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

rfm_scaled_df = pd.DataFrame(
    rfm_scaled,
    index = rfm.index,
    columns = ['Recency', 'Frequency', 'Monetary']
)

rfm_scaled_df.describe()

In [ ]:
sns.boxplot(rfm['Monetary'])

## 3. Elbow Method

In [ ]:
# Range Jumlah Cluster (K)
k_range = range(2,11)

inertia_list = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_scaled_df)
    inertia_list.append(kmeans.inertia_)

# Plot Elbow
plt.figure(figsize=(8,5))
plt.plot(k_range, inertia_list, marker='o', linestyle='--', color='red')
plt.title('Elbow Method')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia / SSE')
plt.xticks(k_range)
plt.grid(True)
plt.show()

## 4. Silhouette Score

In [ ]:
X = rfm_scaled_df

silhouette_scores = []
K = range(2, 11)

print("Silhouette Score untuk setiap jumlah cluster:")
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"K = {k}: Silhouette Score = {score:.4f}")

# Plot Grafik Silhouette Score
plt.figure(figsize=(8,5))
plt.plot(K, silhouette_scores, marker='o', linestyle='--', color='red')
plt.title('Silhouette Score VS Jumlah Cluster (K)')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

## 5. CLUSTERING (K=4)

In [ ]:
# k=4 karena dengan nilai silhouette tertinggi
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled_df)

rfm

In [ ]:
rfm.groupby('Cluster').mean()

### Analisis Cluster

**Cluster 2 — High-Value Customers / VIP**
- Recency sangat rendah → baru saja bertransaksi
- Frequency sangat tinggi → pelanggan sangat sering berbelanja
- Monetary sangat tinggi → nilai pembelian terbesar

**Cluster 3 — Loyal Mid-Value Customers**
- Recency rendah → pelanggan aktif
- Frequency cukup tinggi
- Monetary tinggi tapi di bawah VIP

**Cluster 0 — Active Low-Spenders**
- Recency menengah → masih aktif
- Frequency rendah
- Monetary menengah

**Cluster 1 — Inactive / Lost Customers**
- Recency sangat tinggi → sudah lama tidak berbelanja
- Frequency sangat rendah
- Monetary rendah

### Visualisasi Cluster

In [ ]:
plt.figure(figsize=(16,8))

plt.subplot(131)
sns.boxplot(x='Cluster', y='Recency', data=rfm)
plt.title("Recency per Cluster")

plt.subplot(132)
sns.boxplot(x='Cluster', y='Frequency', data=rfm)
plt.title("Frequency per Cluster")

plt.subplot(133)
sns.boxplot(x='Cluster', y='Monetary', data=rfm)
plt.title("Monetary per Cluster")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16,8))

plt.subplot(131)
sns.scatterplot(x='Recency', y='Frequency', hue='Cluster', data=rfm)
plt.title("Recency vs Frequency")

plt.subplot(132)
sns.scatterplot(x='Recency', y='Monetary', hue='Cluster', data=rfm)
plt.title("Recency vs Monetary")

plt.subplot(133)
sns.scatterplot(x='Frequency', y='Monetary', hue='Cluster', data=rfm)
plt.title("Frequency vs Monetary")

plt.tight_layout()
plt.show()

### PCA Visualization

In [ ]:
# PCA
pca = PCA(n_components=2)
pca_res = pca.fit_transform(rfm_scaled_df)

rfm['pca1'] = pca_res[:,0]
rfm['pca2'] = pca_res[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(x='pca1', y='pca2', hue='Cluster', data=rfm, palette='viridis')
plt.title('PCA - Customer Segments')
plt.show()

## 6. Simpan Model dan Scaler

In [ ]:
import joblib
joblib.dump(kmeans, 'kmeans_rfm_model.pkl')
joblib.dump(scaler, 'rfm_scaler.pkl')

print("Model tersimpan: kmeans_rfm_model.pkl")
print("Scaler tersimpan: rfm_scaler.pkl")

## 7. Simpan Data Preprocessed

In [ ]:
# Simpan data preprocessed
df_final2.to_csv('OnlineRetail_preprocessed.csv', index=False)
print("Data preprocessed tersimpan: OnlineRetail_preprocessed.csv")

# Simpan RFM
rfm.to_csv('OnlineRetail_RFM.csv')
print("Data RFM tersimpan: OnlineRetail_RFM.csv")

## 8. Log ke MLflow / DagsHub

In [ ]:
!pip install mlflow dagshub

In [ ]:
import mlflow
import dagshub

# Hubungkan ke DagsHub
dagshub.init(repo_owner='IslahulHadi', repo_name='Retail-OpsML-Ahul', mlflow=True)

with mlflow.start_run(run_name="KMeans_Segmentation_With_Metrics"):
    # Training
    kmeans_final = KMeans(n_clusters=4, random_state=42)
    kmeans_final.fit(rfm_scaled_df)

    # Hitung metrik
    score = silhouette_score(rfm_scaled_df, kmeans_final.labels_)

    # Log Parameter
    mlflow.log_param("n_clusters", 4)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_init", 10)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("features", "Recency, Frequency, Monetary")

    # Log Metrik
    mlflow.log_metric("silhouette_score", score)
    mlflow.log_metric("inertia", kmeans_final.inertia_)
    mlflow.log_metric("n_customers", rfm.shape[0])

    # Log Model
    mlflow.sklearn.log_model(kmeans_final, "kmeans_rfm_model")

    print(f"Berhasil! Silhouette Score: {score:.4f}")
    print("Cek tab 'Experiments' di DagsHub Anda.")